# Text-to-Text Alignment on LibriSpeech: Silence-Conditioned Whisper States → SONAR

**The missing cell of a 2×2 experiment grid.** The project now has three alignment experiments that
differ along two axes — *calibration corpus* and *decoder conditioning*:

| | silence-conditioned (text-only) | audio-conditioned |
|---|---|---|
| **WikiText** | Exp-01 (`whisper_sonar_alignment_experiment.ipynb`) ✅ | — (no audio exists) |
| **LibriSpeech dev** | **this notebook** | `whisper_sonar_audio_alignment_experiment.ipynb` |

Same recipe as Exp-01 (teacher-forcing over a silent spectrogram, cached encoder output), but the
sentences are **LibriSpeech dev-clean/dev-other transcripts** — the project's actual domain. What the
grid buys, once all three runs exist:

- **Corpus effect** = this notebook vs Exp-01 (conditioning held fixed at silence): does
  domain-matched calibration text improve the map? → spec §10.3, cleanly isolated.
- **Conditioning effect** = the audio notebook vs this one (corpus held fixed at LibriSpeech): what
  does real audio change — overall quality, best layer, early-prefix behavior?
- **Cross-transfer** (Section 8 here + Section 8 there): each fitted matrix evaluated on the other
  conditioning's states — how brittle is `W` to the state distribution it meets at runtime?

**No audio is decoded anywhere in this notebook** — the audio column is stripped from the stream, so
it runs fast even on CPU (SONAR embedding is the main cost). Data discipline as always: **TUNE
shards only** (frozen `sorted → Random(42).shuffle → 25%` rule), DEVTEST/test splits untouched.


## Section 0.1 — Dependencies

In [ ]:
# Run once, then restart the kernel.
# %pip install -U torch transformers datasets scikit-learn scipy pandas matplotlib
# %pip install sonar-space


## Section 0.2 — Imports, seed, device

In [ ]:
import os, math, random, re, json
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("Using device:", DEVICE)


## Section 0.3 — Configuration

Mirrors the audio notebook wherever a knob is shared, so the conditioning comparison is
apples-to-apples:
- `n_per_split = 1000` (FAST 150), 80/10/10 partition, same Ridge grid, same prefix settings.
- `w_wikitext_path` — Exp-01's artifact (WikiText/silence); `w_audio_path` +
  `audio_meta_path` — the audio notebook's artifacts. Each transfer comparison is skipped gracefully
  if the file is absent on this machine.
- Text side uses the project's shared normalization (lowercase `[a-z' ]`) for **both** Whisper
  teacher-forcing and SONAR — identical to the audio notebook, deliberately different from Exp-01's
  raw cased WikiText (that difference is part of the corpus axis and is called out in the rubric).


In [ ]:
FAST = (DEVICE == "cpu")

CONFIG = {
    "whisper_model": "openai/whisper-base",
    "splits": {"dev-clean": ("clean", "validation"), "dev-other": ("other", "validation")},
    "tune_frac": 0.25,
    "n_per_split": 150 if FAST else 1000,
    "batch_size": 16,
    "test_frac": 0.10,
    "val_frac": 0.10,
    "ridge_alphas": [1e-2, 1e-1, 1.0, 10.0, 100.0, 1000.0],
    "prefix_fracs": [0.25, 0.50, 0.75, 1.00],
    "prefix_probe_n": 100 if FAST else 150,
    "w_wikitext_path": "whisper_to_sonar_W.pt",
    "w_audio_path": "whisper_to_sonar_W_audio.pt",
    "audio_meta_path": "audio_alignment_meta.json",
}
print("FAST:", FAST)
CONFIG


## Section 1 — LibriSpeech TUNE-shard transcripts (text only)

Line-by-line:
1. Stream each split with `remove_columns(["audio"])` — no audio bytes are ever decoded.
2. Full ID list → frozen shard rule → TUNE 25%. Printed shard SHAs must match Phase 1 / the audio
   notebook for the same splits (the standing cross-notebook consistency check).
3. Deterministic cap to `n_per_split`: TUNE rows sorted by ID, seeded shuffle, first N — **the same
   selection order as the audio notebook**, so at equal `n_per_split` the analyzed utterance sets
   coincide except for the handful the audio notebook dropped for length (>29 s needs audio to
   detect; the rubric notes this residual difference).
4. Store raw text + normalized text; the normalized string is what both models consume.


In [ ]:
from datasets import load_dataset
import hashlib

def norm(text):
    text = text.lower()
    text = re.sub(r"[^a-z' ]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def sha(obj):
    return hashlib.sha256(json.dumps(sorted(obj)).encode()).hexdigest()[:16]

def shard_ids(ids, tune_frac, seed):
    ordered = sorted(ids)
    random.Random(seed).shuffle(ordered)
    k = int(len(ordered) * tune_frac)
    return set(ordered[:k]), set(ordered[k:])

DATA = {}
for split_name, (hf_config, hf_split) in CONFIG["splits"].items():
    stream = load_dataset("openslr/librispeech_asr", hf_config,
                          split=hf_split, streaming=True).remove_columns(["audio"])
    id_text = [(s["id"], s["text"]) for s in stream]
    tune, _ = shard_ids([i for i, _ in id_text], CONFIG["tune_frac"], SEED)
    rows = sorted([(i, t) for i, t in id_text if i in tune])
    random.Random(SEED).shuffle(rows)
    rows = rows[:CONFIG["n_per_split"]]
    DATA[split_name] = [{"id": i, "text": t, "text_norm": norm(t)} for i, t in rows]
    print(f"{split_name}: {len(id_text)} utts, TUNE {len(tune)} (sha {sha(tune)}), "
          f"analyzed {len(DATA[split_name])}")


## Section 2 — Silence-conditioned decoder states (Exp-01's recipe, LibriSpeech text)

Byte-for-byte the Exp-01 extraction: 30 s of zeros through the **feature extractor** (never a raw
zeros tensor), encoder run **once** and its output reused for every batch via `encoder_outputs=`
(the big speed-up — the decoder pass is all that runs per batch), teacher-forced normalized
transcript, per-layer masked mean-pool over text-token positions (skip the 4 prompt specials and
`<|endoftext|>`), L2-normalize. Output per split: `[n_layers+1, N, 512]`.


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained(CONFIG["whisper_model"])
whisper = (WhisperForConditionalGeneration
           .from_pretrained(CONFIG["whisper_model"]).to(DEVICE).eval())
tok = processor.tokenizer
tok.set_prefix_tokens(language="english", task="transcribe")
N_PREFIX = len(tok.prefix_tokens)

silence = np.zeros(16000 * 30, dtype=np.float32)
sil_feats = processor.feature_extractor(
    silence, sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE)
with torch.no_grad():
    SIL_ENC = whisper.get_encoder()(sil_feats).last_hidden_state
print("cached silence encoder states:", tuple(SIL_ENC.shape))

@torch.no_grad()
def text_states(items, batch_size):
    chunks = []
    for i in range(0, len(items), batch_size):
        b = items[i:i + batch_size]
        enc = tok([x["text_norm"] for x in b], return_tensors="pt", padding=True)
        ids, attn = enc.input_ids.to(DEVICE), enc.attention_mask.to(DEVICE)
        out = whisper(encoder_outputs=(SIL_ENC.expand(ids.shape[0], -1, -1),),
                      decoder_input_ids=ids, output_hidden_states=True, return_dict=True)
        hs = torch.stack(out.decoder_hidden_states)
        mask = attn.clone()
        mask[:, :N_PREFIX] = 0
        mask.scatter_(1, attn.sum(1, keepdim=True) - 1, 0)
        mask = mask.unsqueeze(0).unsqueeze(-1).float()
        pooled = (hs * mask).sum(2) / mask.sum(2).clamp(min=1.0)
        chunks.append(F.normalize(pooled, dim=-1).float().cpu())
        if (i // batch_size) % 10 == 0:
            print(f"  batch {i // batch_size + 1}/{math.ceil(len(items) / batch_size)}")
    return torch.cat(chunks, dim=1)

STATES = {}
for split_name, items in DATA.items():
    print(f"extracting {split_name} ({len(items)} transcripts)...")
    STATES[split_name] = text_states(items, CONFIG["batch_size"])
N_LAYERS = next(iter(STATES.values())).shape[0]
print({s: tuple(v.shape) for s, v in STATES.items()})


## Section 3 — SONAR embeddings of the same transcripts

In [ ]:
from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline

t2vec = TextToEmbeddingModelPipeline(encoder="text_sonar_basic_encoder",
                                     tokenizer="text_sonar_basic_encoder",
                                     device=torch.device("cpu"))

Y = {}
for split_name, items in DATA.items():
    with torch.no_grad():
        e = t2vec.predict([x["text_norm"] for x in items],
                          source_lang="eng_Latn", batch_size=64).float()
    Y[split_name] = F.normalize(e, dim=-1).cpu()
    print(split_name, tuple(Y[split_name].shape))


## Section 4 — Train / val / test partition (same rule as the audio notebook)

In [ ]:
PART = {}
for split_name, items in DATA.items():
    idx = list(range(len(items)))
    random.Random(SEED).shuffle(idx)
    n = len(idx)
    n_test, n_val = int(n * CONFIG["test_frac"]), int(n * CONFIG["val_frac"])
    PART[split_name] = {"test": idx[:n_test],
                        "val": idx[n_test:n_test + n_val],
                        "train": idx[n_test + n_val:]}
    print(split_name, {k: len(v) for k, v in PART[split_name].items()},
          f"chance top-1 = {100.0 / max(n_test, 1):.2f}%")


## Section 5 — Fitting and evaluation machinery (identical to Exp-01 / audio notebook)

In [ ]:
def fit_ols(X, Yt):
    Wm, *_ = np.linalg.lstsq(X, Yt, rcond=None)
    return Wm

def evaluate_map(Wm, X, Yt):
    P = F.normalize(torch.from_numpy(X).float() @ torch.from_numpy(Wm).float(), dim=-1)
    T = F.normalize(torch.from_numpy(Yt).float(), dim=-1)
    sim = P @ T.T
    d = sim.diag()
    n = sim.shape[0]
    rank = (sim > d.unsqueeze(1)).sum(dim=1)
    off = (sim.sum() - d.sum()) / (n * n - n)
    return {"top1": (rank < 1).float().mean().item(),
            "top5": (rank < 5).float().mean().item(),
            "top10": (rank < 10).float().mean().item(),
            "mean_rank": rank.float().mean().item() + 1.0,
            "paired_cos": d.mean().item(),
            "random_cos": off.item()}

def fit_ridge(Xtr, Ytr, Xval, Yval, alphas):
    from sklearn.linear_model import Ridge
    best = (None, None, -1.0)
    for a in alphas:
        reg = Ridge(alpha=a, fit_intercept=False).fit(Xtr, Ytr)
        Wm = reg.coef_.T
        v = evaluate_map(Wm, Xval, Yval)["top1"]
        if v > best[2]:
            best = (Wm, a, v)
    return best

print("machinery ready")


## Section 6 — Layer sweep: fit `W_text_ls`, evaluate per split

One matrix per (layer, method), fitted on both splits' train sets jointly, evaluated on each split's
own test pool — same protocol as the audio notebook so the two sweeps overlay directly.


In [ ]:
rows, fitted = [], {}
for layer in range(N_LAYERS):
    Xtr = np.concatenate([STATES[s][layer].numpy()[PART[s]["train"]] for s in DATA])
    Ytr = np.concatenate([Y[s].numpy()[PART[s]["train"]] for s in DATA])
    Xval = np.concatenate([STATES[s][layer].numpy()[PART[s]["val"]] for s in DATA])
    Yval = np.concatenate([Y[s].numpy()[PART[s]["val"]] for s in DATA])

    W_ols = fit_ols(Xtr, Ytr)
    W_rdg, alpha, _ = fit_ridge(Xtr, Ytr, Xval, Yval, CONFIG["ridge_alphas"])
    fitted[layer] = {"ols": W_ols, "ridge": W_rdg, "alpha": alpha}

    for s in DATA:
        Xte = STATES[s][layer].numpy()[PART[s]["test"]]
        Yte = Y[s].numpy()[PART[s]["test"]]
        for mname, Wm in [("OLS", W_ols), (f"Ridge(α={alpha:g})", W_rdg)]:
            rows.append({"layer": layer, "method": mname, "split": s,
                         **{k: round(v, 4) for k, v in evaluate_map(Wm, Xte, Yte).items()}})
    done = [r for r in rows if r["layer"] == layer and r["method"].startswith("Ridge")]
    print(f"layer {layer}: " + "  ".join(f"{r['split']} top1={r['top1']:.1%}" for r in done))

results = pd.DataFrame(rows)
summary = (results.groupby(["layer", "method"])["top1"].mean()
           .reset_index().sort_values("top1", ascending=False))
summary.head(8)


## Section 7 — Best configuration + shuffled-pairs control

In [ ]:
best_row = summary.iloc[0]
BEST_LAYER = int(best_row["layer"])
BEST_METHOD = best_row["method"]
W_TEXT_LS = fitted[BEST_LAYER]["ridge" if BEST_METHOD.startswith("Ridge") else "ols"]
print(f"best: layer {BEST_LAYER}, {BEST_METHOD}, mean test top1 = {best_row['top1']:.1%}")

Xtr = np.concatenate([STATES[s][BEST_LAYER].numpy()[PART[s]["train"]] for s in DATA])
Ytr = np.concatenate([Y[s].numpy()[PART[s]["train"]] for s in DATA])
Xval = np.concatenate([STATES[s][BEST_LAYER].numpy()[PART[s]["val"]] for s in DATA])
Yval = np.concatenate([Y[s].numpy()[PART[s]["val"]] for s in DATA])
perm = np.random.RandomState(SEED).permutation(len(Ytr))
W_ctrl, _, _ = fit_ridge(Xtr, Ytr[perm], Xval, Yval, CONFIG["ridge_alphas"])
for s in DATA:
    Xte = STATES[s][BEST_LAYER].numpy()[PART[s]["test"]]
    Yte = Y[s].numpy()[PART[s]["test"]]
    m = evaluate_map(W_ctrl, Xte, Yte)
    print(f"shuffled control on {s}: top1={m['top1']:.2%} "
          f"(chance {1.0 / len(PART[s]['test']):.2%})")


## Section 8 — Cross-matrix comparison on these text-conditioned states

Evaluate every available matrix on **this notebook's** test pools (text-conditioned LibriSpeech
states):

- `W_text_ls` (this run) — the in-distribution reference;
- `W_wikitext` (Exp-01) — same conditioning, different corpus → isolates the **corpus effect**;
- `W_audio` (audio notebook) — same corpus, different conditioning → conditioning mismatch measured
  from the text side (the audio notebook measures the mirror direction).

Each external matrix is evaluated at its own native layer (Exp-01: 4; audio: from its meta JSON) and
at this run's best layer. Missing artifacts are skipped with a notice.


In [ ]:
cross_rows = []
externals = []
if os.path.exists(CONFIG["w_wikitext_path"]):
    externals.append(("W_wikitext(Exp-01)",
                      torch.load(CONFIG["w_wikitext_path"], map_location="cpu").float().numpy(), 4))
else:
    print("Exp-01 artifact not found — corpus-effect comparison skipped.")
if os.path.exists(CONFIG["w_audio_path"]):
    a_layer = 4
    if os.path.exists(CONFIG["audio_meta_path"]):
        a_layer = json.load(open(CONFIG["audio_meta_path"])).get("best_layer", 4)
    externals.append(("W_audio(exp-04)",
                      torch.load(CONFIG["w_audio_path"], map_location="cpu").float().numpy(), a_layer))
else:
    print("Audio-notebook artifact not found — conditioning comparison skipped.")

for s in DATA:
    Yte = Y[s].numpy()[PART[s]["test"]]
    Xte_best = STATES[s][BEST_LAYER].numpy()[PART[s]["test"]]
    cross_rows.append({"split": s, "matrix": "W_text_ls(this nb)",
                       "eval_layer": BEST_LAYER,
                       **{k: round(v, 4) for k, v in evaluate_map(W_TEXT_LS, Xte_best, Yte).items()}})
    for name, Wm, native_layer in externals:
        for layer in sorted({native_layer, BEST_LAYER}):
            Xte = STATES[s][layer].numpy()[PART[s]["test"]]
            cross_rows.append({"split": s, "matrix": name, "eval_layer": layer,
                               **{k: round(v, 4) for k, v in evaluate_map(Wm, Xte, Yte).items()}})
cross_df = pd.DataFrame(cross_rows)
cross_df


## Section 9 — Prefix probe (silence-conditioned, LibriSpeech text)

Same probe as always at the best layer with `W_text_ls`. The interesting overlay (Section 10): this
curve vs Exp-01's WikiText/silence curve (corpus effect on early prefixes) vs the audio notebook's
curve when available (conditioning effect — audio gives the decoder whole-utterance evidence from
step one, silence cannot).


In [ ]:
@torch.no_grad()
def token_states_one(item, layer):
    ids = tok(item["text_norm"], return_tensors="pt").input_ids.to(DEVICE)
    out = whisper(encoder_outputs=(SIL_ENC,), decoder_input_ids=ids,
                  output_hidden_states=True, return_dict=True)
    return out.decoder_hidden_states[layer][0][N_PREFIX:-1].float().cpu()

W_t = torch.from_numpy(W_TEXT_LS).float()
probe_rows = []
for s in DATA:
    test_idx = PART[s]["test"][:CONFIG["prefix_probe_n"]]
    pool = F.normalize(Y[s][PART[s]["test"]].float(), dim=-1)
    own = {orig: k for k, orig in enumerate(PART[s]["test"])}
    token_cache = [(own[i], token_states_one(DATA[s][i], BEST_LAYER)) for i in test_idx]
    for frac in CONFIG["prefix_fracs"]:
        preds, owns = [], []
        for pos, ts in token_cache:
            k = max(1, math.ceil(frac * ts.shape[0]))
            preds.append(F.normalize(ts[:k].mean(0, keepdim=True), dim=-1))
            owns.append(pos)
        P = F.normalize(torch.cat(preds) @ W_t, dim=-1)
        sim = P @ pool.T
        d = sim[torch.arange(len(owns)), torch.tensor(owns)]
        rank = (sim > d.unsqueeze(1)).sum(1)
        probe_rows.append({"split": s, "prefix": f"{int(frac*100)}%",
                           "top1": round((rank < 1).float().mean().item(), 4),
                           "top5": round((rank < 5).float().mean().item(), 4)})
prefix_df = pd.DataFrame(probe_rows)
prefix_df.pivot(index="prefix", columns="split", values="top1")


## Section 10 — Visual summary

Layer sweep per split; prefix curves overlaid on Exp-01's WikiText/silence reference (and, if you
paste them in after running the audio notebook, its audio-conditioned curve); cross-matrix top-1
bars when external artifacts were found.


In [ ]:
n_panels = 3 if len(cross_df["matrix"].unique()) > 1 else 2
fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 4.2))

for s in DATA:
    g = results[(results["split"] == s) & (results["method"].str.startswith("Ridge"))] \
        .sort_values("layer")
    axes[0].plot(g["layer"], g["top1"] * 100, marker="o", label=s)
axes[0].set_xlabel("decoder layer"); axes[0].set_ylabel("test top-1 (%)")
axes[0].set_title("W_text_ls quality by layer (Ridge)"); axes[0].legend()

EXP01_PREFIX = {"25%": 0.29, "50%": 0.765, "75%": 0.975, "100%": 1.00}
AUDIO_PREFIX = None   # paste the audio notebook's per-split means here after its run, e.g. {"25%": ..., ...}
for s in DATA:
    g = prefix_df[prefix_df["split"] == s]
    axes[1].plot(g["prefix"], g["top1"], marker="o", label=f"{s} (LS text)")
axes[1].plot(list(EXP01_PREFIX.keys()), list(EXP01_PREFIX.values()),
             "k--", marker="x", label="Exp-01 (WikiText)")
if AUDIO_PREFIX:
    axes[1].plot(list(AUDIO_PREFIX.keys()), list(AUDIO_PREFIX.values()),
                 "r:", marker="s", label="audio-conditioned")
axes[1].set_xlabel("prefix seen"); axes[1].set_ylabel("top-1")
axes[1].set_title("Prefix probe: corpus (and conditioning) overlays"); axes[1].legend()

if n_panels == 3:
    piv = (cross_df.sort_values("top1", ascending=False)
           .drop_duplicates(["split", "matrix"])       # best eval_layer per matrix
           .pivot(index="split", columns="matrix", values="top1"))
    piv.plot.bar(ax=axes[2], rot=0)
    axes[2].set_ylabel("test top-1")
    axes[2].set_title("Cross-matrix on text-conditioned states")
plt.tight_layout(); plt.show()


## Section 11 — Artifacts

In [ ]:
torch.save(torch.from_numpy(W_TEXT_LS).float(), "whisper_to_sonar_W_text_librispeech.pt")
results.to_csv("text_ls_alignment_results.csv", index=False)
prefix_df.to_csv("text_ls_alignment_prefix.csv", index=False)
cross_df.to_csv("text_ls_alignment_cross.csv", index=False)
meta = {
    "config": {k: v for k, v in CONFIG.items() if k != "splits"},
    "best_layer": BEST_LAYER, "best_method": str(BEST_METHOD),
    "mean_test_top1": float(best_row["top1"]),
    "utterance_ids": {s: [x["id"] for x in DATA[s]] for s in DATA},
    "partitions": {s: {k: [DATA[s][i]["id"] for i in v] for k, v in PART[s].items()}
                   for s in DATA},
}
with open("text_ls_alignment_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print("Saved: whisper_to_sonar_W_text_librispeech.pt, "
      "text_ls_alignment_{results,prefix,cross}.csv, text_ls_alignment_meta.json")


## Section 12 — Reading the results (the 2×2 view)

Fill the grid with mean test top-1 (and, more discriminative, the 25%-prefix top-1):

| | silence | audio |
|---|---|---|
| **WikiText** | Exp-01 | — |
| **LibriSpeech** | this run | audio notebook |

- **Corpus effect** (this run vs Exp-01, both silence): if LibriSpeech-text calibration clearly
  beats WikiText on these pools, spec §10.3 resolves toward domain-matched text regardless of
  conditioning. Caveat: Exp-01 used raw cased WikiText and a 400 pool; compare against chance
  levels, and note the normalization difference is part of the corpus axis here.
- **Conditioning effect** (audio notebook vs this run, both LibriSpeech): differences in best layer,
  overall top-1, and especially the 25%-prefix point tell you how much the deployment distribution
  (audio-conditioned states) differs geometrically from cheap text-only calibration.
- **Cross-matrix table (Section 8):** the practical question — *how much does a mismatched `W` cost?*
  If `W_wikitext` ≈ `W_text_ls` here **and** the audio notebook's transfer showed `W_silence` ≈
  `W_audio` there, the map is robust and the cheapest calibration wins. If mismatches are expensive,
  the spec should mandate audio-conditioned LibriSpeech calibration (`W_audio`) as the frozen
  artifact.
- **Controls:** shuffled ≈ chance everywhere, or stop and debug before interpreting anything.

**Wiki follow-up:** file results into the 2×2 discussion — this run as part of
`exp-04`'s page (or its own `exp-05-text-ls-alignment.md` if run separately), update
`whisper-sonar-linear-map.md` §10.3 status and `overview.md`, append a log entry.
